# Skill Dimension Loader

Maintains the `warehouse.dim_skill` dimension table with incremental refresh capability.

## Purpose
Track canonical skills from governed metadata taxonomy for consistent skill tagging.

## Key Features
* Stable surrogate keys for skills (auto-increment)
* Skill categories and sector mappings
* Skill aliases for fuzzy matching
* Technical vs soft skill classification
* Demand scoring for skill prioritization
* SCD Type 1 (overwrite on change)
* Idempotent: safe to re-run

## Architecture
**Source**: Metadata CSV files (`/Workspace/.../LMIP/metadata/canonical_skills.csv`)  
**Target**: `workspace.warehouse.dim_skill`  
**Metadata**: `workspace.metadata.dim_skill_refresh_log`  
**Mode**: Incremental (merge new/updated skills)

## Batch Processing
* Tracks refresh history in metadata table
* Auto-assigns surrogate keys to new skills
* Updates existing skills with latest taxonomy
* Validates data quality after each refresh

In [0]:
dbutils.widgets.dropdown("force_full_refresh", "false", ["true", "false"], "Force Full Refresh")
dbutils.widgets.text("metadata_path", "/Workspace/Users/aaryan.shrivastav1403@gmail.com/LMIP/metadata", "Metadata Path")

FORCE_FULL_REFRESH = dbutils.widgets.get("force_full_refresh") == "true"
METADATA_PATH = dbutils.widgets.get("metadata_path")

In [0]:
from datetime import datetime
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, TimestampType, BooleanType, LongType, DoubleType
import json

CATALOG = "workspace"
WAREHOUSE_SCHEMA = f"{CATALOG}.warehouse"
METADATA_SCHEMA = f"{CATALOG}.metadata"

TARGET_TABLE = f"{WAREHOUSE_SCHEMA}.dim_skill"
METADATA_TABLE = f"{METADATA_SCHEMA}.dim_skill_refresh_log"

run_id = datetime.now().strftime("%Y%m%d_%H%M%S")
run_timestamp = datetime.now()

print(f"Run ID: {run_id}")
print(f"Metadata path: {METADATA_PATH}")
print(f"Force full refresh: {FORCE_FULL_REFRESH}")

In [0]:
%sql
-- Create skill dimension table if not exists
CREATE TABLE IF NOT EXISTS workspace.warehouse.dim_skill (
  skill_sk BIGINT NOT NULL COMMENT 'Surrogate key for skill',
  skill_id STRING NOT NULL COMMENT 'Natural key from taxonomy',
  skill_name STRING NOT NULL COMMENT 'Canonical skill name',
  skill_category STRING NOT NULL COMMENT 'Skill category',
  sector_key STRING NOT NULL COMMENT 'Associated sector',
  skill_level INT NOT NULL COMMENT 'Skill proficiency level',
  parent_skill_id STRING COMMENT 'Parent skill for hierarchies',
  aliases ARRAY<STRING> COMMENT 'Skill aliases for matching',
  is_technical BOOLEAN NOT NULL COMMENT 'Is technical skill',
  demand_score DOUBLE NOT NULL COMMENT 'Demand metric score',
  is_active BOOLEAN NOT NULL COMMENT 'Is skill active',
  updated_at TIMESTAMP NOT NULL COMMENT 'Last update timestamp',
  CONSTRAINT pk_dim_skill PRIMARY KEY (skill_sk)
)
USING DELTA
COMMENT 'Canonical skill dimension from governed taxonomy';

-- Create metadata tracking table
CREATE TABLE IF NOT EXISTS workspace.metadata.dim_skill_refresh_log (
  run_id STRING,
  skills_extracted INT,
  skills_inserted INT,
  skills_updated INT,
  force_full_refresh BOOLEAN,
  processed_at TIMESTAMP,
  status STRING,
  error_message STRING
)
USING DELTA
COMMENT 'Tracks skill dimension refresh history';

In [0]:
print("Loading skill taxonomy from metadata files...", end=" ")

# Load skill taxonomy
skills_df = spark.read.csv(
    f"{METADATA_PATH}/canonical_skills.csv",
    header=True,
    inferSchema=True
)

# Transform for warehouse dimension
skill_extract_df = skills_df.select(
    F.col("skill_key").alias("skill_id"),
    F.col("canonical_skill").alias("skill_name"),
    F.col("skill_category"),
    F.coalesce(F.col("sector_key"), F.lit("CROSS_SECTOR")).alias("sector_key"),
    F.lit(1).alias("skill_level"),
    F.lit(None).cast("string").alias("parent_skill_id"),
    F.when(F.col("skill_category") == "Technical", True).otherwise(False).alias("is_technical"),
    F.lit(0.0).alias("demand_score"),
    F.lit(True).alias("is_active"),
    F.split(F.col("aliases"), "\\|").alias("aliases")
)

skills_count = skill_extract_df.count()
print(f"✓ Loaded {skills_count} skills from taxonomy")

# Get current max surrogate key
max_sk_result = spark.sql(f"SELECT COALESCE(MAX(skill_sk), 0) as max_sk FROM {TARGET_TABLE}").collect()
max_sk = max_sk_result[0]['max_sk']

print(f"Current max surrogate key: {max_sk}")

In [0]:
# Define metadata schema
metadata_schema = StructType([
    StructField("run_id", StringType(), True),
    StructField("skills_extracted", IntegerType(), True),
    StructField("skills_inserted", IntegerType(), True),
    StructField("skills_updated", IntegerType(), True),
    StructField("force_full_refresh", BooleanType(), True),
    StructField("processed_at", TimestampType(), True),
    StructField("status", StringType(), True),
    StructField("error_message", StringType(), True)
])

try:
    print(f"Processing skills into {TARGET_TABLE}...", end=" ")
    
    # Check if table schema matches expected schema FIRST
    existing_cols = [row.col_name for row in spark.sql(f"DESCRIBE {TARGET_TABLE}").collect()]
    expected_cols = ['skill_sk', 'skill_id', 'skill_name', 'skill_category', 'sector_key', 'skill_level', 'parent_skill_id', 'aliases', 'is_technical', 'demand_score', 'is_active', 'updated_at']
    schema_matches = set(existing_cols) == set(expected_cols)
    
    # Only query existing skills if schema matches
    if schema_matches:
        existing_skills = spark.sql(f"SELECT skill_id, skill_sk FROM {TARGET_TABLE}")
        
        # Join to assign keys (existing or new)
        from pyspark.sql.window import Window
        
        skills_with_keys = skill_extract_df.alias("s").join(
            existing_skills.alias("e"),
            F.col("s.skill_id") == F.col("e.skill_id"),
            "left"
        )
        
        # Assign surrogate keys
        window_spec = Window.orderBy("s.skill_id")
        
        skills_final = skills_with_keys.withColumn(
            "skill_sk",
            F.coalesce(F.col("e.skill_sk"), F.lit(max_sk) + F.row_number().over(window_spec))
        ).withColumn(
            "updated_at",
            F.lit(run_timestamp)
        ).select(
            F.col("skill_sk").cast(LongType()),
            F.col("s.skill_id").alias("skill_id"),
            F.col("s.skill_name").alias("skill_name"),
            F.col("s.skill_category").alias("skill_category"),
            F.col("s.sector_key").alias("sector_key"),
            F.col("s.skill_level").alias("skill_level"),
            F.col("s.parent_skill_id").alias("parent_skill_id"),
            F.col("s.aliases").alias("aliases"),
            F.col("s.is_technical").alias("is_technical"),
            F.col("s.demand_score").alias("demand_score"),
            F.col("s.is_active").alias("is_active"),
            "updated_at"
        )
    else:
        # Schema mismatch: assign keys without joining to existing table
        from pyspark.sql.window import Window
        window_spec = Window.orderBy("skill_id")
        
        skills_final = skill_extract_df.withColumn(
            "skill_sk",
            (F.lit(max_sk) + F.row_number().over(window_spec)).cast(LongType())
        ).withColumn(
            "updated_at",
            F.lit(run_timestamp)
        ).select(
            "skill_sk",
            "skill_id",
            "skill_name",
            "skill_category",
            "sector_key",
            "skill_level",
            "parent_skill_id",
            "aliases",
            "is_technical",
            "demand_score",
            "is_active",
            "updated_at"
        )
    
    # Create temp view for merge
    skills_final.createOrReplaceTempView("skills_to_merge")
    
    if FORCE_FULL_REFRESH or not schema_matches:
        # Full refresh: drop and recreate with new schema
        if not schema_matches:
            print(f"Schema mismatch detected. Performing full refresh...")
            spark.sql(f"DROP TABLE IF EXISTS {TARGET_TABLE}")
            spark.sql(f"""
            CREATE TABLE {TARGET_TABLE} (
              skill_sk BIGINT NOT NULL COMMENT 'Surrogate key for skill',
              skill_id STRING NOT NULL COMMENT 'Natural key from taxonomy',
              skill_name STRING NOT NULL COMMENT 'Canonical skill name',
              skill_category STRING NOT NULL COMMENT 'Skill category',
              sector_key STRING NOT NULL COMMENT 'Associated sector',
              skill_level INT NOT NULL COMMENT 'Skill proficiency level',
              parent_skill_id STRING COMMENT 'Parent skill for hierarchies',
              aliases ARRAY<STRING> COMMENT 'Skill aliases for matching',
              is_technical BOOLEAN NOT NULL COMMENT 'Is technical skill',
              demand_score DOUBLE NOT NULL COMMENT 'Demand metric score',
              is_active BOOLEAN NOT NULL COMMENT 'Is skill active',
              updated_at TIMESTAMP NOT NULL COMMENT 'Last update timestamp',
              CONSTRAINT pk_dim_skill PRIMARY KEY (skill_sk)
            )
            USING DELTA
            COMMENT 'Canonical skill dimension from governed taxonomy'
            """)
        else:
            spark.sql(f"TRUNCATE TABLE {TARGET_TABLE}")
        
        skills_final.write.format("delta").mode("append").saveAsTable(TARGET_TABLE)
        skills_inserted = skills_final.count()
        skills_updated = 0
        print(f"✓ Full refresh: {skills_inserted} skills inserted")
    else:
        # Incremental: merge
        merge_sql = f"""
        MERGE INTO {TARGET_TABLE} target
        USING skills_to_merge source
        ON target.skill_id = source.skill_id
        WHEN MATCHED THEN UPDATE SET
            target.skill_name = source.skill_name,
            target.skill_category = source.skill_category,
            target.sector_key = source.sector_key,
            target.skill_level = source.skill_level,
            target.parent_skill_id = source.parent_skill_id,
            target.aliases = source.aliases,
            target.is_technical = source.is_technical,
            target.demand_score = source.demand_score,
            target.is_active = source.is_active,
            target.updated_at = source.updated_at
        WHEN NOT MATCHED THEN INSERT *
        """
        
        spark.sql(merge_sql)
        
        # Count metrics
        skills_inserted = spark.sql(f"""
            SELECT COUNT(*) as cnt FROM skills_to_merge
            WHERE skill_id NOT IN (SELECT skill_id FROM {TARGET_TABLE})
        """).collect()[0]['cnt']
        
        skills_updated = skills_count - skills_inserted
        
        print(f"✓ Merge complete: {skills_inserted} new, {skills_updated} updated")
    
    # Log to metadata
    metadata_data = [(
        run_id,
        skills_count,
        skills_inserted,
        skills_updated,
        FORCE_FULL_REFRESH,
        run_timestamp,
        'success',
        None
    )]
    
    metadata_record = spark.createDataFrame(metadata_data, schema=metadata_schema)
    metadata_record.write.format("delta").mode("append").saveAsTable(METADATA_TABLE)
    
    result = {
        "status": "success",
        "run_id": run_id,
        "skills_extracted": skills_count,
        "skills_inserted": skills_inserted,
        "skills_updated": skills_updated,
        "target_table": TARGET_TABLE,
        "metadata_table": METADATA_TABLE
    }
    
    print(json.dumps(result, indent=2))
    
except Exception as e:
    error_msg = str(e)
    print(f"✗ Error: {error_msg}")
    
    # Log failure to metadata
    metadata_data = [(
        run_id,
        skills_count if 'skills_count' in locals() else 0,
        0,
        0,
        FORCE_FULL_REFRESH,
        run_timestamp,
        'failed',
        error_msg
    )]
    
    metadata_record = spark.createDataFrame(metadata_data, schema=metadata_schema)
    metadata_record.write.format("delta").mode("append").saveAsTable(METADATA_TABLE)
    
    raise

In [0]:
%sql
-- Validate skill dimension
SELECT 
  COUNT(*) as total_skills,
  COUNT(DISTINCT skill_category) as skill_categories,
  COUNT(DISTINCT sector_key) as sectors,
  SUM(CASE WHEN is_technical THEN 1 ELSE 0 END) as technical_skills,
  SUM(CASE WHEN NOT is_technical THEN 1 ELSE 0 END) as soft_skills,
  AVG(demand_score) as avg_demand_score,
  SUM(CASE WHEN is_active THEN 1 ELSE 0 END) as active_skills
FROM workspace.warehouse.dim_skill;

-- Sample skills by category
SELECT 
  skill_sk,
  skill_name,
  skill_category,
  sector_key,
  is_technical,
  demand_score,
  SIZE(aliases) as alias_count,
  is_active,
  updated_at
FROM workspace.warehouse.dim_skill
ORDER BY skill_category, skill_name
LIMIT 25;

-- Show refresh history
SELECT 
  run_id,
  skills_extracted,
  skills_inserted,
  skills_updated,
  force_full_refresh,
  processed_at,
  status
FROM workspace.metadata.dim_skill_refresh_log
ORDER BY processed_at DESC
LIMIT 10;